In [7]:

import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from data.queries import engine, Session
from data.models import Paper
from typing import Optional
from sqlalchemy import func

# inster .. imports possible



def get_study_by(attr: str, value: int | str, after_2025: bool=False) -> dict:
    session = Session()
    try:
        if after_2025:
            query = session.query(Paper).filter(
                Paper.entrez_year > 2025
            )
        else:
            query = session.query(Paper).filter(
                (Paper.entrez_year.is_(None)) | (Paper.entrez_year <= 2025)
            )
        query = query.filter(getattr(Paper, attr) == value)

        # if there is several papers raise an error
        papers = query.all()
        if len(papers) > 1:
            raise ValueError(f"Multiple papers found for {attr} = {value}")

        paper = query.first()
        if not paper:
            return {}
        else:
            return {
                "id": paper.id,
                "pubmed_id": paper.pubmed_id,
                "doi": paper.doi,
                "title": paper.title,
                "abstract": paper.abstract,
            }
    finally:
        session.close()


def get_study_by_title(title: str, include_substring: bool = False, after_2025: bool=False) -> list[dict]:
    title = title.lower()
    session = Session()
    try:
        if include_substring:
            query = session.query(Paper).filter(
                func.lower(Paper.title).contains(func.lower(title))
            )
        else:
            query = session.query(Paper).filter(
                func.lower(Paper.title) == func.lower(title)
            )
        if after_2025:
            query = query.filter(
                Paper.entrez_year > 2025
            )
        else:
            query = query.filter(
                (Paper.entrez_year.is_(None)) | (Paper.entrez_year <= 2025)
            )
        papers = query.all()

        if not papers:
            return []

        result = []
        for paper in papers:
            result.append({
                "id": paper.id,
                "pubmed_id": paper.pubmed_id,
                "doi": paper.doi,
                "title": paper.title,
                "abstract": paper.abstract,
            })

        return result

    finally:
        session.close()


def read_in_asreview() -> list[pd.DataFrame]:
    input_file = "/home/veral/PsyNamic/data/raw_data/asreview_dataset_all_Psychedelic Study.csv"
    df = pd.read_csv(input_file)
    df['title'] = df['title'].str.lower()
    included = df[df['included'] == 1]
    excluded = df[df['included'] == 0]
    #    array([ 1., nan,  0.])
    # lower title
    excluded_after_stopping = df[df['included'].isna()]

    return [included, excluded, excluded_after_stopping, df]


def read_in_all_retrieved_studies() -> list[pd.DataFrame]:
    input_dir = '/home/veral/PsyNamic/PsyNamic-Webapp/data/relevant_studies/studies_20260320_00-09-37.csv'
    df = pd.read_csv(input_dir)
    # remove all papers that have entrez_date > 2025
    df['entrez_date'] = pd.to_datetime(df['entrez_year'], errors='coerce')
    df = df[df['entrez_date'] < '2025-01-01']

    included = df.loc[df['prediction'] == 1].copy()
    excluded = df.loc[df['prediction'] == 0].copy()
    # convert title to lowercase
    included.loc[:, 'title'] = included['title'].str.lower()
    excluded.loc[:, 'title'] = excluded['title'].str.lower()
    return [included, excluded]


def find_study_in_db(pmid: int, doi: str, title: str, after_2025: bool = False, rel_pmid1: Optional[int] = None, rel_pmid2: Optional[int] = None) -> bool:
    study_data = None
    if pmid:
        study_data = get_study_by('pubmed_id', pmid, after_2025=after_2025)
    if study_data:
        return True
    else:
        if rel_pmid1:
            study_data = get_study_by('pubmed_id', rel_pmid1, after_2025=after_2025)
        if study_data:
            return True
        if rel_pmid2:
            study_data = get_study_by('pubmed_id', rel_pmid2, after_2025=after_2025)
        if study_data:
            return True
        if doi:
            doi = str(doi).replace('https://doi.org/', '')
            study_data = get_study_by('doi', doi, after_2025=after_2025)
        if study_data:
            return True
        else:
            study_data = get_study_by_title(title, after_2025=after_2025)
            if study_data:
                return True
            else:
                return False        




In [9]:
articles = "/home/veral/PsyNamic/PsyNamic-Webapp/validation/psynamic_validation_included_articles.csv"
reviews = "/home/veral/PsyNamic/PsyNamic-Webapp/validation/psynamic_validation_sr_library.csv"

df_articles = pd.read_csv(articles, delimiter=';')
df_articles.replace('nA', None, inplace=True)

df_reviews = pd.read_csv(reviews, delimiter=';')
df_reviews.replace('nA', None, inplace=True)

included_man, excluded_man, excluded_after_stopping_man, df = read_in_asreview()
included_auto, excluded_auto = read_in_all_retrieved_studies()

How many articles and systematic review are there in the set?

In [12]:
print(len(df_articles))
print(len(df_reviews))

413
30


How many systematic reviews?

In [13]:
nr_studie_in_db = 0

df_articles['in_psynamic'] = False
df_articles['in_excluded'] = False
df_articles['in_excluded_after_stopping'] = False
df_articles['excluded_by_bert'] = False
df_articles['in_later_retrieved'] = False

for index, row in df_articles.iterrows():
    title = row['Title'].lower()
    if find_study_in_db(row['PMID'], row['doi'], row['Title'], after_2025=False, rel_pmid1=row['Related Publication PMID 1'], rel_pmid2=row['Related Publication PMID 2']):
        df_articles.at[index, 'in_psynamic'] = True

    elif title in excluded_man['title'].values:
        df_articles.at[index, 'in_excluded'] = True

    elif title in excluded_after_stopping_man['title'].values:
        df_articles.at[index, 'in_excluded_after_stopping'] = True

    if title in excluded_auto['title'].values:
        df_articles.at[index, 'excluded_by_bert'] = True

    if find_study_in_db(row['PMID'], row['doi'], row['Title'], after_2025=True, rel_pmid1=row['Related Publication PMID 1'], rel_pmid2=row['Related Publication PMID 2']):
        df_articles.at[index, 'in_later_retrieved'] = True

Number of articles excluded by BERT?

In [15]:
df_articles['excluded_by_bert'].sum()


np.int64(0)

How large is the overlap between the articles (with duplicates)?

In [16]:
nr_articles = df_articles['in_psynamic'].sum()
(nr_articles / len(df_articles)) * 100

np.float64(84.26150121065376)

Percentage of articles from RS that are included in PsyNamic database (up until 2025. no duplicates)?

In [19]:
df_articles_no_duplicates = df_articles.drop_duplicates(subset=['PMID', 'doi', 'Title'], keep='first')
print(len(df_articles_no_duplicates))
nr_articles_no_duplicate = df_articles_no_duplicates['in_psynamic'].sum()
print(nr_articles_no_duplicate)

322
262


In [22]:
# get number of articles where either in_psyanamic, in_excluded or in_excluded_after_stopping is True
df_articles_no_duplicates['in_psynamic_or_excluded'] = df_articles_no_duplicates['in_psynamic'] | df_articles_no_duplicates['in_excluded'] | df_articles_no_duplicates['in_excluded_after_stopping']
nr_articles_no_duplicate_or_excluded = df_articles_no_duplicates['in_psynamic_or_excluded'].sum()
print(nr_articles_no_duplicate_or_excluded)
(nr_articles_no_duplicate_or_excluded / len(df_articles_no_duplicates)) * 100

275


/tmp/ipykernel_220773/3562798738.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_articles_no_duplicates['in_psynamic_or_excluded'] = df_articles_no_duplicates['in_psynamic'] | df_articles_no_duplicates['in_excluded'] | df_articles_no_duplicates['in_excluded_after_stopping']


np.float64(85.40372670807453)

Number of studies from RS that were manually excluded?

In [24]:
print(df_articles_no_duplicates['in_excluded'].sum())
df_articles_no_duplicates[df_articles_no_duplicates['in_excluded'] == True][['SR_No','Title']]

8


,SR_No,Title
54,3,Psychedelic microdosing benefits and challenge...
57,3,Report on psychoactive drug use among adolesce...
65,3,Short-Term Treatment Effects of a Substance Us...
121,6,Time until relapse after augmentation with sin...
225,16,A preliminary investigation of ibogaine: case ...
240,16,"Ibogaine: Complex pharmacokinetics, concerns f..."
325,23,Short-Term Treatment Effects of a Substance Us...
331,24,"Persisting reductions in cannabis, opioid, and..."


Number of studies from RS that were in excluded after stopping criteria?

In [25]:
df_articles_no_duplicates['in_excluded_after_stopping'].sum()

np.int64(5)

Number of studies from RS that were included after 2025?

In [26]:
df_articles_no_duplicates['in_later_retrieved'].sum()

np.int64(0)

What is the average percentage of articles from RS that are included in PsyNamic database (up until 2025)?

In [27]:
sr_nos = df_articles['SR_No'].unique()
# add column overlap
df_reviews['overlap'] = pd.NA
df_reviews['in_psynamic'] = pd.NA
for sr_no in sr_nos:
    total_studies = df_reviews[df_reviews['SR'] == sr_no]['Total Nr'].iloc[0]
    included = df_articles[(df_articles['SR_No'] == sr_no) & (df_articles['in_psynamic'])].shape[0] 
    percentage = included / total_studies * 100 if total_studies > 0 else 0
    df_reviews.loc[df_reviews['SR'] == sr_no, 'overlap'] = percentage
    nr_included = df_articles[(df_articles['SR_No'] == sr_no) & (df_articles['in_psynamic'])].shape[0]
    df_reviews.loc[df_reviews['SR'] == sr_no, 'in_psynamic'] = nr_included
df_reviews[['SR', 'Authors', 'overlap', 'in_psynamic']]

,SR,Authors,overlap,in_psynamic
0,1,A. Bahji; C. A. Zarate; G. H. Vazquez,66.666667,4
1,2,J. J. Breeksema; B. W. Kuin; J. Kamphuis; W. v...,91.111111,41
2,3,J. Calleja-Conde; J. A. Morales-García; V. Ech...,48.148148,13
3,4,E. Capuzzi; A. Caldiroli; M. Capellazzi; I. Ta...,85.714286,6
4,5,V. B. Cavenaghi; L. P. da Costa; A. L. T. Lace...,41.666667,5
5,6,A. A. Conley; A. E. Q. Norwood; T. C. Hatvany;...,84.782609,39
6,7,C. M. H. de Vos; N. L. Mason; K. P. C. Kuypers,20.0,4
7,8,R. Du; R. Han; K. Niu; J. Xu; Z. Zhao; G. Lu; ...,60.0,6
8,9,C. L. Felsch; K. P. C. Kuypers,56.666667,17
9,10,N. L. Galvão-Coelho; W. Marx; M. Gonzalez; J. ...,100.0,12


In [29]:
print(f'Average overlap: {df_reviews["overlap"].mean():.2f}%')
print(f'Median overlap: {df_reviews["overlap"].median():.2f}%')


Average overlap: 77.23%
Median overlap: 85.25%


What's the median of in psynamic and in included?

In [30]:
df_reviews['in_psynamic'].median()

np.float64(9.0)

In [31]:
df_reviews['Included'].median()

np.float64(11.0)

Number of missing articles that don't have a PMID?

In [32]:
df_filtered = df_articles_no_duplicates[
    df_articles_no_duplicates['in_psynamic_or_excluded'] == False
]

# count where the PMID is None
nr_no_pmid = df_filtered['PMID'].isna().sum()
# percentage of articles without PMID
(nr_no_pmid / len(df_filtered)) * 100
nr_no_pmid

np.int64(13)

Random sample of 20 articles that were not found by search string

In [37]:
# check how many of the not included the pmid is none
df_articles_not_found_in_psynamic = df_articles[
    (df_articles['in_psynamic'] == False) &
    (df_articles['in_excluded'] == False) &
    (df_articles['in_excluded_after_stopping'] == False)
]
df_subset = df_articles_not_found_in_psynamic.sample(n=20, random_state=40)

# df_subset[['Title'], ['PMID'], ['doi']]
# write out csv with only these columns
df_subset[['Title', 'PMID', 'doi']].to_csv('/home/veral/PsyNamic/PsyNamic-Webapp/validation/20_random_sample_not_found_in_psynamic.csv', index=False)
